# BO coherence / séries
Vérifier la cohérence du bo_type (1/2/3/5), des scores finaux et du nombre de maps.

In [5]:
import sys
from pathlib import Path
import polars as pl

def _find_root():
    cand = Path.cwd()
    for c in [cand, *cand.parents]:
        if (c / "src").exists() and (c / "data").exists():
            return c
    return cand

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.dota_data import read_processed_tables

tables = read_processed_tables(ROOT / "data" / "processed")
matches = tables["matches"]
series_path = ROOT / "data" / "processed" / "series.parquet"
series_df = pl.read_parquet(series_path) if series_path.exists() else None
series_maps_path = ROOT / "data" / "metrics" / "series_maps.parquet"
series_maps = pl.read_parquet(series_maps_path) if series_maps_path.exists() else None

TARGET_LEAGUE = None  # ex: 16477
TARGET_SERIES = None  # ex: 863552

BO_ALLOWED = {1: {1}, 2: {1, 2}, 3: {2, 3}, 5: {3, 4, 5}}

def maps_ok(bo, count):
    return count in BO_ALLOWED.get(bo, set())


## Distribution des BO et scores

In [6]:
if series_df is None or series_df.is_empty():
    print("series.parquet manquant ou vide (relancer make parquet)")
else:
    dist_bo = series_df.group_by("bo_type").agg(pl.len().alias("series")).sort("bo_type")
    display(dist_bo)

    scores = series_df.select(
        "bo_type",
        "match_count",
        "score_team_a",
        "score_team_b",
        "series_id",
        "leagueid",
    )
    display(scores.head())


bo_type,series
i64,u32
1,1072
2,852
3,4471
5,199


bo_type,match_count,score_team_a,score_team_b,series_id,leagueid
i64,i64,i64,i64,i64,i64
3,3,2,1,1039062,18920
3,3,1,2,1038719,18920
3,3,2,1,1038288,18920
3,2,2,0,1037873,18920
3,2,0,2,1036811,18920


## Anomalies : map_count vs bo_type et équipes

In [7]:
if series_df is None or series_df.is_empty():
    print("series.parquet manquant")
else:
    anomalies = (
        series_df
        .with_columns(pl.struct(["bo_type","match_count"]).map_elements(lambda s: maps_ok(s["bo_type"], s["match_count"])).alias("maps_ok"))
        .with_columns((pl.col("teams_in_series") != 2).alias("bad_teams"))
        .filter((~pl.col("maps_ok")) | pl.col("bad_teams"))
        .sort(["bo_type","match_count","series_id","leagueid"])
    )
    display(anomalies)


/tmp/ipykernel_340622/649738606.py:6: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  .with_columns(pl.struct(["bo_type","match_count"]).map_elements(lambda s: maps_ok(s["bo_type"], s["match_count"])).alias("maps_ok"))


series_id,leagueid,league_name,tournament_name,tournament_slug,tournament_tier,tournament_location,series_type_raw,bo_type,match_count,teams,teams_in_series,team_a,team_b,score_team_a,score_team_b,winner_team_id,max_wins,start_time_min,start_time_max,maps_ok,bad_teams
i64,i64,str,str,str,str,str,i64,i64,i64,list[i64],i64,i64,i64,i64,i64,i64,i64,i64,i64,bool,bool


## Séries par ligue (synthèse anomalies)

In [8]:
if series_df is None or series_df.is_empty():
    print("series.parquet manquant")
else:
    by_league = (
        series_df
        .with_columns(pl.struct(["bo_type","match_count"]).map_elements(lambda s: maps_ok(s["bo_type"], s["match_count"])).alias("maps_ok"))
        .with_columns((pl.col("teams_in_series") != 2).alias("bad_teams"))
        .group_by(["leagueid","league_name","tournament_name"])
        .agg(
            pl.len().alias("series"),
            ((~pl.col("maps_ok")) | pl.col("bad_teams")).cast(pl.Int64).sum().alias("anomaly_series"),
        )
        .sort("anomaly_series", descending=True)
    )
    display(by_league)


/tmp/ipykernel_340622/2931080355.py:6: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  .with_columns(pl.struct(["bo_type","match_count"]).map_elements(lambda s: maps_ok(s["bo_type"], s["match_count"])).alias("maps_ok"))


leagueid,league_name,tournament_name,series,anomaly_series
i64,str,str,u32,i64
17414,"""BLAST SLAM I""","""BLAST SLAM I""",29,0
17056,"""CCT Dota 2 Ser…","""CCT Dota 2 Ser…",29,0
18358,"""PGL Wallachia …","""PGL Wallachia …",47,0
16779,"""Elite League S…","""Elite League S…",3,0
18666,"""CCT Dota 2 Sea…","""CCT Dota 2 Sea…",21,0
…,…,…,…,…
18668,"""FISSURE PLAYGR…","""FISSURE PLAYGR…",14,0
15889,"""TDL Americas P…","""TDL Americas P…",1,0
16935,"""The Internatio…","""The Internatio…",55,0


## Focus filters (league/series) + maps

In [9]:
if series_df is None or series_df.is_empty():
    print("series.parquet manquant")
else:
    filt = series_df
    if TARGET_LEAGUE is not None:
        filt = filt.filter(pl.col("leagueid") == TARGET_LEAGUE)
    if TARGET_SERIES is not None:
        filt = filt.filter(pl.col("series_id") == TARGET_SERIES)
    if filt.is_empty():
        print("Aucune série pour ces filtres")
    else:
        display(filt.sort(["series_id","bo_type"]))
        if series_maps is not None:
            sm = series_maps
            if TARGET_LEAGUE is not None:
                sm = sm.filter(pl.col("leagueid") == TARGET_LEAGUE)
            if TARGET_SERIES is not None:
                sm = sm.filter(pl.col("series_id") == TARGET_SERIES)
            display(sm.sort(["series_id","map_num","start_time"]))
        else:
            print("series_maps.parquet manquant")


series_id,leagueid,league_name,tournament_name,tournament_slug,tournament_tier,tournament_location,series_type_raw,bo_type,match_count,teams,teams_in_series,team_a,team_b,score_team_a,score_team_b,winner_team_id,max_wins,start_time_min,start_time_max
i64,i64,str,str,str,str,str,i64,i64,i64,list[i64],i64,i64,i64,i64,i64,i64,i64,i64,i64
0,18830,"""BLAST Slam V: …","""BLAST Slam V: …","""blast-slam-v-c…","""regular""","""online""",0,1,1,"[9444076, 9315393]",2,9444076,9315393,0,1,9315393,1,1761051142,1761051142
816359,16846,"""FISSURE Univer…","""FISSURE Univer…","""fissure-univer…","""regular""","""online""",1,3,3,"[8728920, 8291895]",2,8728920,8291895,1,2,8291895,2,1724355059,1724363401
838795,16093,"""BetBoom Dacha …","""BetBoom Dacha …","""betboom-dacha-…","""regular""","""online""",1,1,1,"[9081007, 8588969]",2,9081007,8588969,1,0,9081007,1,1704355281,1704355281
838796,16093,"""BetBoom Dacha …","""BetBoom Dacha …","""betboom-dacha-…","""regular""","""online""",1,3,2,"[2576071, 9279613]",2,2576071,9279613,2,0,2576071,2,1704355380,1704358994
838802,16093,"""BetBoom Dacha …","""BetBoom Dacha …","""betboom-dacha-…","""regular""","""online""",1,1,1,"[9081007, 8588969]",2,9081007,8588969,1,0,9081007,1,1704358792,1704358792
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1044594,18866,"""European Pro L…","""European Pro L…","""european-pro-l…","""regular""","""online""",1,3,2,"[9017006, 9303383]",2,9017006,9303383,0,2,9303383,2,1765198839,1765202381
1044694,18866,"""European Pro L…","""European Pro L…","""european-pro-l…","""regular""","""online""",1,3,2,"[9886449, 2576071]",2,9886449,2576071,0,2,2576071,2,1765217016,1765220615
1044854,19054,"""Snake Trophy""","""Snake Trophy""","""snake-trophy""","""regular""","""online""",1,3,3,"[9546449, 9722899]",2,9546449,9722899,2,1,9546449,2,1765256683,1765263782


series_id,leagueid,series_type,bo_type,map_num,match_id,start_time,radiant_team_id,dire_team_id,radiant_win
i64,i64,i64,i64,i64,i64,i64,i64,i64,bool
0,18830,1,1,1,8521719394,1761051142,9444076,9315393,false
816359,16846,3,3,1,7909186596,1724355059,8728920,8291895,true
816359,16846,3,3,2,7909275659,1724358805,8291895,8728920,true
816359,16846,3,3,3,7909343508,1724363401,8728920,8291895,false
838795,16093,1,1,1,7520706781,1704355281,9081007,8588969,true
…,…,…,…,…,…,…,…,…,…
1044854,19054,3,3,3,8597003724,1765263782,9546449,9722899,true
1044898,18866,3,3,1,8597139842,1765274867,9895247,9600141,true
1044898,18866,3,3,2,8597191886,1765278054,9895247,9600141,true


### Diagnostic : séries malformées
Comptage des séries écartées par les règles BO (BO inconnu, trop de maps, mauvais nombre d'équipes, série incomplète/à égalité).

In [10]:

from pathlib import Path
import polars as pl

matches_path = Path("../data/processed/matches.parquet")
if not matches_path.exists():
    matches_path = Path("data/processed/matches.parquet")
matches = pl.read_parquet(matches_path)

series_matches = matches.filter(pl.col("series_id").is_not_null())

bo_max = {1: 1, 2: 2, 3: 3, 5: 5}
required = {1: 1, 2: 1, 3: 2, 5: 3}

series = (
    series_matches
    .group_by(["series_id", "leagueid"])
    .agg(
        pl.first("bo_type").alias("bo_type"),
        pl.first("series_type").alias("series_type"),
        pl.len().alias("maps"),
        pl.concat_list(["radiant_team_id", "dire_team_id"]).list.unique().list.len().alias("teams"),
    )
    .with_columns(pl.coalesce([pl.col("bo_type"), pl.col("series_type")]).alias("bo"))
    .with_columns(
        pl.col("bo").replace(bo_max).alias("max_maps"),
        pl.col("bo").replace(required).alias("need_wins"),
    )
)

def pair_set(df: pl.DataFrame) -> set[tuple[int, int]]:
    return {(int(r["series_id"]), int(r["leagueid"])) for r in df.iter_rows(named=True)}

too_many = series.filter(pl.col("max_maps").is_not_null() & (pl.col("maps") > pl.col("max_maps")))
bad_bo = series.filter(pl.col("bo").is_null())
bad_teams = series.filter(pl.col("teams") != 2)

matches_with_bo = series_matches.with_columns(pl.coalesce([pl.col("bo_type"), pl.col("series_type")]).alias("bo"))

incomplete_rows = []
for df in matches_with_bo.partition_by(["series_id", "leagueid"], as_dict=False, maintain_order=True):
    b = df["bo"][0]
    need = required.get(b)
    if need is None:
        continue
    rad_w = int(df["radiant_win"].sum())
    dire_w = df.height - rad_w
    if max(rad_w, dire_w) < need or rad_w == dire_w:
        incomplete_rows.append({
            "series_id": int(df["series_id"][0]),
            "leagueid": int(df["leagueid"][0]),
            "bo": b,
            "maps": df.height,
            "rad_wins": rad_w,
            "dire_wins": dire_w,
        })

if incomplete_rows:
    incomplete = pl.DataFrame(incomplete_rows)
else:
    incomplete = pl.DataFrame(
        [],
        schema={
            "series_id": pl.Int64,
            "leagueid": pl.Int64,
            "bo": pl.Int64,
            "maps": pl.Int64,
            "rad_wins": pl.Int64,
            "dire_wins": pl.Int64,
        },
    )

total_series = series.height
anom_ids = pair_set(too_many) | pair_set(bad_bo) | pair_set(bad_teams) | pair_set(incomplete)

summary = pl.DataFrame(
    [
        {"bucket": "total_series", "count": total_series},
        {"bucket": "kept_clean", "count": total_series - len(anom_ids)},
        {"bucket": "too_many_maps", "count": too_many.height},
        {"bucket": "unknown_bo", "count": bad_bo.height},
        {"bucket": "bad_team_count", "count": bad_teams.height},
        {"bucket": "incomplete_or_draw", "count": incomplete.height},
    ]
)

summary
too_many.head(20)
bad_bo.head(20)
bad_teams.head(20)
incomplete.head(20)


series_id,leagueid,bo,maps,rad_wins,dire_wins
i64,i64,i64,i64,i64,i64
1037873,18920,3,2,1,1
1036811,18920,3,2,1,1
1028251,18863,3,2,1,1
1027411,18863,3,2,1,1
1027046,18863,3,2,1,1
…,…,…,…,…,…
989168,18358,3,2,1,1
987941,18358,3,2,1,1
980308,18111,3,2,1,1
